# 04 — MLflow Experiment Analysis

Compare all training runs across experiments using the MLflow Python client:
- Load all runs from the `stock-predictor` experiment
- Plot accuracy & F1 over retraining runs
- Hyperparameter vs performance table
- Highlight the best run

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'ml-backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import mlflow
from mlflow.tracking import MlflowClient
from datetime import datetime

sns.set_theme(style='darkgrid')

# Point to local mlruns directory
MLRUNS_DIR = os.path.join('..', 'ml-backend', 'mlruns')
mlflow.set_tracking_uri(f'file:///{os.path.abspath(MLRUNS_DIR).replace(os.sep, "/")}')
client = MlflowClient()
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

## 1. List All Experiments

In [ ]:
experiments = client.search_experiments()
print(f'Found {len(experiments)} experiments:')
for exp in experiments:
    runs = client.search_runs(experiment_ids=[exp.experiment_id])
    print(f'  [{exp.experiment_id}] {exp.name}  — {len(runs)} runs')

## 2. Load All Stock Predictor Runs

In [ ]:
all_runs = mlflow.search_runs(experiment_names=['stock-predictor'])

if all_runs.empty:
    print('No runs found. Train the model first (run notebook 02 or POST /retrain).')
else:
    # Tidy columns
    metric_cols = [c for c in all_runs.columns if c.startswith('metrics.')]
    param_cols  = [c for c in all_runs.columns if c.startswith('params.')]
    display_cols = ['run_id', 'start_time', 'status'] + metric_cols + param_cols
    print(f'Total runs: {len(all_runs)}')
    print(all_runs[display_cols].to_string(index=False))

## 3. Metrics Over Time

In [ ]:
if not all_runs.empty and 'metrics.f1_score' in all_runs.columns:
    runs_sorted = all_runs.sort_values('start_time').reset_index(drop=True)
    run_nums = range(1, len(runs_sorted) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # F1 score
    ax = axes[0]
    f1_vals = runs_sorted['metrics.f1_score'].astype(float)
    ax.plot(run_nums, f1_vals, 'o-', color='steelblue', linewidth=2, markersize=8)
    ax.axhline(f1_vals.max(), color='green', linestyle='--', linewidth=1, label=f'Best F1={f1_vals.max():.4f}')
    ax.set_title('F1 Score Across Retraining Runs')
    ax.set_xlabel('Run #')
    ax.set_ylabel('F1 Score')
    ax.set_xticks(list(run_nums))
    ax.legend()

    # Accuracy
    ax = axes[1]
    if 'metrics.accuracy' in runs_sorted.columns:
        acc_vals = runs_sorted['metrics.accuracy'].astype(float)
        ax.plot(run_nums, acc_vals, 'o-', color='tomato', linewidth=2, markersize=8)
        ax.axhline(acc_vals.max(), color='green', linestyle='--', linewidth=1, label=f'Best Acc={acc_vals.max():.4f}')
        ax.set_title('Accuracy Across Retraining Runs')
        ax.set_xlabel('Run #')
        ax.set_ylabel('Accuracy')
        ax.set_xticks(list(run_nums))
        ax.legend()

    plt.suptitle('Model Performance Over Retraining History', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough data to plot — need at least one completed run.')

## 4. Hyperparameter vs F1 Score

In [ ]:
if not all_runs.empty and 'metrics.f1_score' in all_runs.columns:
    param_metric_cols = [c for c in all_runs.columns if c.startswith('params.')] + ['metrics.f1_score', 'metrics.accuracy']
    table = all_runs[param_metric_cols].copy()
    table.columns = [c.replace('params.', '').replace('metrics.', '') for c in table.columns]
    table = table.sort_values('f1_score', ascending=False)

    # Highlight best row
    print('Hyperparameter vs Performance Table:')
    print(table.to_string(index=False))

    # Scatter: n_estimators vs f1
    if 'n_estimators' in table.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        scatter = ax.scatter(
            table['n_estimators'].astype(float),
            table['f1_score'].astype(float),
            c=table['f1_score'].astype(float),
            cmap='RdYlGn', s=100, edgecolors='k', linewidths=0.5
        )
        plt.colorbar(scatter, ax=ax, label='F1 Score')
        ax.set_xlabel('n_estimators')
        ax.set_ylabel('F1 Score')
        ax.set_title('n_estimators vs F1 Score')
        plt.tight_layout()
        plt.show()

## 5. Best Run Highlight

In [ ]:
if not all_runs.empty and 'metrics.f1_score' in all_runs.columns:
    best_idx = all_runs['metrics.f1_score'].astype(float).idxmax()
    best_run = all_runs.loc[best_idx]
    print('🏆 Best Run:')
    print(f'  Run ID:    {best_run["run_id"]}')
    print(f'  Start:     {best_run["start_time"]}')
    for col in [c for c in all_runs.columns if c.startswith('metrics.')]:
        print(f'  {col.replace("metrics.", ""):20s}: {float(best_run[col]):.4f}')
    for col in [c for c in all_runs.columns if c.startswith('params.')]:
        print(f'  {col.replace("params.", ""):20s}: {best_run[col]}')

## 6. All Experiments Summary

In [ ]:
for exp_name in ['stock-predictor', 'anomaly-detector', 'risk-scorer']:
    try:
        runs = mlflow.search_runs(experiment_names=[exp_name])
        print(f'\n=== {exp_name} ({len(runs)} runs) ===')
        if not runs.empty:
            metric_cols = [c for c in runs.columns if c.startswith('metrics.')]
            print(runs[metric_cols].describe().round(4).to_string())
    except Exception as e:
        print(f'{exp_name}: {e}')